In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:

df = pd.read_csv(r"C:\Users\hp\Downloads\ML\spam.csv")
df.head()


,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
df.shape

(5572, 2)

In [4]:
df['Category'].value_counts()

Category
ham     4825
spam     747
Name: count, dtype: int64

In [6]:
def preprocess_text(text):

    words = word_tokenize(text)    
    words = [word.lower() for word in words if word.isalpha()]   
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]       
    stemmer = PorterStemmer()
    words = [stemmer.stem(word) for word in words]
    return " ".join(words)

In [7]:
df["Message"] = df["Message"].apply(preprocess_text)

In [9]:
x = df['Message']
y = df['Category']

x_train, x_test, y_train, y_test = train_test_split(x, y,
    test_size=0.2,
    random_state=42
)

print(f"Training data size: {len(x_train)}")
print(f"Test data size: {len(x_test)}")

vectorizer = TfidfVectorizer(max_features=5000)
x_train_tf = vectorizer.fit_transform(x_train)
x_test_tf = vectorizer.transform(x_test)

Training data size: 4457
Test data size: 1115


In [10]:
model = MultinomialNB()
model.fit(x_train_tf, y_train)

print("✅ Model trained successfully!")

✅ Model trained successfully!


In [11]:
y_pred = model.predict(x_test_tf)
accuracy = accuracy_score(y_test, y_pred)

print("=" * 55)
print("   SPAM CLASSIFICATION RESULTS")
print("=" * 55)
print(f"\n🎯 Accuracy: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"\n--- Detailed Classification Report ---")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
print("--- Confusion Matrix ---")
print(f"                Predicted Ham  Predicted Spam")
print(f"Actual Ham      {cm[0][0]:>10}    {cm[0][1]:>10}")
print(f"Actual Spam     {cm[1][0]:>10}    {cm[1][1]:>10}")

   SPAM CLASSIFICATION RESULTS

🎯 Accuracy: 0.9668 (96.7%)

--- Detailed Classification Report ---
              precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       0.99      0.76      0.86       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.96      1115

--- Confusion Matrix ---
                Predicted Ham  Predicted Spam
Actual Ham             965             1
Actual Spam             36           113


In [12]:
def predict_email(text):
    stop_words = set(stopwords.words('english'))
    cleaned = ' '.join(word.lower() for word in word_tokenize(text) if word.isalpha())
    cleaned = ' '.join(word for word in cleaned.split() if word not in stop_words)

    features = vectorizer.transform([cleaned])
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0]

    emoji = "✅" if prediction == "ham" else "🚫"
    print(f"{emoji} Prediction: {prediction.upper()}")
    print(f"   Confidence — Ham: {probability[0]:.2%}, Spam: {probability[1]:.2%}")
    return prediction

In [14]:
print("--- Testing Custom Emails ---\n")

print("Email 1:")
print("Hey, are we still meeting for lunch tomorrow?")
predict_email("Hey, are we still meeting for lunch tomorrow?")

print("\nEmail 2:")
print("CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!")
predict_email("CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!")

print("\nEmail 3:")
print("Please find the quarterly report attached as discussed.")
predict_email("Please find the quarterly report attached as discussed.")

--- Testing Custom Emails ---

Email 1:
Hey, are we still meeting for lunch tomorrow?
✅ Prediction: HAM
   Confidence — Ham: 99.57%, Spam: 0.43%

Email 2:
CONGRATULATIONS! You won a FREE iPhone! Click NOW to claim!
🚫 Prediction: SPAM
   Confidence — Ham: 21.38%, Spam: 78.62%

Email 3:
Please find the quarterly report attached as discussed.
✅ Prediction: HAM
   Confidence — Ham: 88.43%, Spam: 11.57%


np.str_('ham')